# 01 · Tokenization

> **Objectives**
> - What tokenization is and *why* it matters for LLMs
> - How Byte-Pair Encoding (BPE) works — from theory to implementation
> - How to use real tokenizers (GPT-2, LLaMA, Mistral)
> - Engineering pitfalls: context limits, cost estimation, multilingual efficiency
> - Hands-on experiments you can run right now

---


## 1. Overview

LLMs do not read text — they read **tokens**.

A **token** is the atomic unit of input and output for any transformer-based language model.
Before any text reaches the model, it must be converted into a sequence of integer IDs. After generation, those IDs must be decoded back into text.

```
"Hello, world!"
      ↓  tokenizer.encode()
[15496, 11, 995, 0]
      ↓  model processes integers
[15496, 11, 995, 0, ...]
      ↓  tokenizer.decode()
"Hello, world! ..."
```

Understanding tokenization is foundational because:
- It determines the **effective context length** of your model
- It affects **cost** (you pay per token in API calls)
- It explains many **model failure modes** (e.g. arithmetic, spelling)
- It impacts **multilingual performance** dramatically


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install transformers tiktoken sentencepiece -q

import re
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("Core libraries ready")


In [ ]:
# Load real tokenizers
from transformers import AutoTokenizer

gpt2_tokenizer   = AutoTokenizer.from_pretrained("gpt2")
llama_tokenizer  = AutoTokenizer.from_pretrained("huggyllama/llama-7b")   # or use local path
mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

print("Vocab sizes:")
print(f"  GPT-2   : {gpt2_tokenizer.vocab_size:,}")
print(f"  LLaMA   : {llama_tokenizer.vocab_size:,}")
print(f"  Mistral : {mistral_tokenizer.vocab_size:,}")


## 3. Background Concepts

### 3.1 Why not just use characters or words?

| Approach | Pros | Cons |
|---|---|---|
| **Character-level** | No OOV words; tiny vocab | Very long sequences; slow training |
| **Word-level** | Intuitive | Huge vocab; OOV problem; morphology blind |
| **Subword (BPE)** | Balances vocab size & sequence length | Requires training; language-dependent |

Modern LLMs use **subword tokenization** — typically BPE or a variant of it.

---

### 3.2 Byte-Pair Encoding (BPE) — The Math

BPE was originally a data compression algorithm (Gage, 1994), adapted for NLP by Sennrich et al. (2016).

**Algorithm:**

Given a corpus $C$, a target vocabulary size $V$, and an initial vocabulary of individual characters $\Sigma$:

1. Count all adjacent symbol pairs in $C$
2. Find the most frequent pair $(a, b)$
3. Merge every occurrence of $(a, b)$ → $ab$ in $C$
4. Add $ab$ to vocabulary
5. Repeat until $|V|$ merges are done

**Formal merge rule:**

$$
\text{merge}(a, b) = \underset{(a,b) \in P}{\arg\max} \ \text{count}(a, b, C)
$$

where $P$ is the set of all adjacent symbol pairs in corpus $C$.

The final vocabulary contains:
- All individual bytes/characters (fallback)
- All learned merge results

This guarantees **zero out-of-vocabulary** tokens (any text can be encoded byte by byte as a fallback).


## 4. BPE Implementation from Scratch

In [ ]:
def get_vocab(corpus: list[str]) -> dict:
    """
    Build initial vocabulary from a corpus.
    Each word is represented as a tuple of characters + end-of-word marker </w>.
    """
    vocab = Counter()
    for word in corpus:
        # Split into chars, add end-of-word marker
        chars = tuple(list(word) + ["</w>"])
        vocab[chars] += 1
    return dict(vocab)


def get_pairs(vocab: dict) -> dict:
    """Count all adjacent symbol pairs across the vocabulary."""
    pairs = Counter()
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return pairs


def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """Merge all occurrences of `pair` in the vocabulary."""
    new_vocab = {}
    bigram = pair[0] + pair[1]
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab


def train_bpe(corpus: list[str], num_merges: int) -> list[tuple]:
    """Train BPE and return the list of merge rules."""
    vocab = get_vocab(corpus)
    merges = []

    print(f"Initial vocab size: {len(set(sym for word in vocab for sym in word))}")
    print(f"Training {num_merges} merges...\n")

    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
        if i < 10 or i % 10 == 0:
            print(f"  Merge {i+1:3d}: {best[0]!r} + {best[1]!r} → {best[0]+best[1]!r}  "
                  f"(count={pairs[best]})")

    return merges, vocab


# ── Run it ──
corpus = [
    "low", "low", "low", "low", "low",
    "lower", "lower",
    "newest", "newest", "newest", "newest", "newest", "newest",
    "widest", "widest", "widest",
]

merges, final_vocab = train_bpe(corpus, num_merges=15)
print("\nFinal vocabulary entries:")
for word, freq in sorted(final_vocab.items(), key=lambda x: -x[1]):
    print(f"  {word}  (freq={freq})")


In [ ]:
def bpe_encode(word: str, merges: list[tuple]) -> list[str]:
    """Encode a word using learned BPE merge rules."""
    tokens = list(word) + ["</w>"]
    for merge in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == merge[0] and tokens[i+1] == merge[1]:
                new_tokens.append(merge[0] + merge[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


# Test encoding
test_words = ["low", "lower", "newest", "widest", "lowest"]
print("BPE Encoding results:")
print("-" * 40)
for w in test_words:
    tokens = bpe_encode(w, merges)
    print(f"  {w:12s} → {tokens}")


## 5. Real Tokenizers in Practice

In [ ]:
# ── Basic encode / decode ──────────────────────────────────────────────
text = "The quick brown fox jumps over the lazy dog."

tokens = gpt2_tokenizer.encode(text)
decoded = gpt2_tokenizer.decode(tokens)

print(f"Input : {text}")
print(f"Tokens: {tokens}")
print(f"Count : {len(tokens)}")
print(f"Decoded: {decoded}")
print()

# Show each token with its string representation
print("Token breakdown:")
for tok_id in tokens:
    tok_str = gpt2_tokenizer.decode([tok_id])
    print(f"  {tok_id:6d}  →  {tok_str!r}")


In [ ]:
# ── Visualise token boundaries ─────────────────────────────────────────
def visualize_tokens(text: str, tokenizer, title: str = ""):
    tokens = tokenizer.encode(text)
    token_strings = [tokenizer.decode([t]) for t in tokens]

    colors = plt.cm.Set3(np.linspace(0, 1, len(tokens)))

    fig, ax = plt.subplots(figsize=(14, 2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    x = 0.01
    for i, (tok_str, color) in enumerate(zip(token_strings, colors)):
        width = max(len(tok_str) * 0.013, 0.03)
        rect = mpatches.FancyBboxPatch(
            (x, 0.25), width, 0.5,
            boxstyle="round,pad=0.01",
            facecolor=color, edgecolor="gray", linewidth=0.8
        )
        ax.add_patch(rect)
        ax.text(x + width / 2, 0.5, tok_str.replace(" ", "·"),
                ha="center", va="center", fontsize=8, fontweight="bold")
        ax.text(x + width / 2, 0.15, str(tokens[i]),
                ha="center", va="center", fontsize=6, color="gray")
        x += width + 0.005

    ax.set_title(f"{title}  |  {len(tokens)} tokens", fontsize=11)
    plt.tight_layout()
    plt.show()


sample = "Tokenization is the foundation of every LLM system."
visualize_tokens(sample, gpt2_tokenizer, "GPT-2")


## 6. Engineering Insights

### 6.1 Context Length is in Tokens, Not Characters

A model with a 4096-token context window does **not** mean 4096 characters.
English text averages ~4 characters per token, but this varies widely.

```
English prose   ≈ 0.75 words/token  (≈ 4 chars/token)
Python code     ≈ varies, often < 4 chars/token
JSON / XML      ≈ verbose, many tokens
CJK languages   ≈ 1–2 chars/token (much worse ratio)
```

**Engineering implication:** Always measure context usage in tokens, not characters or words.


In [ ]:
# ── Context length experiment ──────────────────────────────────────────
samples = {
    "English prose":   "The transformer architecture revolutionized natural language processing.",
    "Python code":     "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "JSON":            '{"name": "Alice", "age": 30, "city": "Sydney", "role": "engineer"}',
    "Chinese":         "变形金刚架构彻底改变了自然语言处理领域的发展方向。",
    "Arabic":          "غيّرت بنية المحوّل مجال معالجة اللغة الطبيعية تغييراً جذرياً.",
}

print(f"{'Type':<20} {'Chars':>6} {'Tokens':>7} {'Chars/Token':>12}")
print("-" * 48)
for label, text in samples.items():
    n_chars = len(text)
    n_tokens = len(gpt2_tokenizer.encode(text))
    ratio = n_chars / n_tokens
    print(f"{label:<20} {n_chars:>6} {n_tokens:>7} {ratio:>12.2f}")


### 6.2 Token Cost Estimation

API providers (OpenAI, Anthropic, etc.) charge **per token**.

A quick estimator:

$$
\text{cost} = (n_{\text{input}} \times p_{\text{in}} + n_{\text{output}} \times p_{\text{out}}) \times 10^{-6}
$$

where prices are in USD per million tokens.


In [ ]:
def estimate_cost(text_in: str, text_out: str, tokenizer,
                  price_in_per_M: float = 3.0,   # e.g. GPT-4o input
                  price_out_per_M: float = 15.0): # e.g. GPT-4o output
    n_in  = len(tokenizer.encode(text_in))
    n_out = len(tokenizer.encode(text_out))
    cost  = (n_in * price_in_per_M + n_out * price_out_per_M) / 1_000_000
    print(f"Input tokens  : {n_in:,}")
    print(f"Output tokens : {n_out:,}")
    print(f"Estimated cost: ${cost:.6f} USD")
    return cost

system_prompt = "You are a helpful assistant that answers concisely."
user_message  = "Explain the difference between RAG and fine-tuning for LLMs."
model_output  = "RAG retrieves relevant context at inference time; fine-tuning bakes knowledge into weights."

estimate_cost(system_prompt + user_message, model_output, gpt2_tokenizer)


### 6.3 Why LLMs Struggle with Arithmetic and Spelling

A common question: *Why can't GPT do simple arithmetic?*

The answer is partly tokenization. Numbers are often split in non-intuitive ways.


In [ ]:
# ── Tokenization artifacts ─────────────────────────────────────────────
edge_cases = [
    "1234567890",
    "0.1 + 0.2 = 0.3",
    "SolidGoldMagikarp",         # Famous GPT-2 anomaly token
    "    ",                      # Whitespace
    "Hello
world",
    "color colour",              # British vs American spelling
]

print("Edge case tokenization (GPT-2):")
print("-" * 50)
for text in edge_cases:
    tokens = gpt2_tokenizer.encode(text)
    token_strs = [gpt2_tokenizer.decode([t]) for t in tokens]
    print(f"  Input  : {text!r}")
    print(f"  Tokens : {token_strs}")
    print()


## 7. Tokenizer Comparison: GPT-2 vs LLaMA vs Mistral

In [ ]:
comparison_texts = [
    "Hello, how are you?",
    "def quicksort(arr): return arr if len(arr) <= 1 else ...",
    "The mitochondria is the powerhouse of the cell.",
    "1 + 1 = 2, therefore 100 + 200 = 300",
]

tokenizers = {
    "GPT-2":   gpt2_tokenizer,
    "LLaMA":   llama_tokenizer,
    "Mistral": mistral_tokenizer,
}

print(f"{'Text':<45} {'GPT-2':>6} {'LLaMA':>6} {'Mistral':>8}")
print("-" * 70)
for text in comparison_texts:
    row = f"{text[:43]:<45}"
    for name, tok in tokenizers.items():
        n = len(tok.encode(text))
        row += f"{n:>7}"
    print(row)


## 8. Exercises

Try these on your own to deepen your understanding:

1. **BPE scaling:** Increase `num_merges` in Section 4 to 50. How does the vocabulary evolve? At what point do whole words become single tokens?

2. **Token efficiency:** Encode a paragraph of code and the same paragraph of English prose. Which is more token-efficient? Why?

3. **Cost calculator:** Modify the cost estimator to accept a system prompt that's reused across 10,000 calls. How much does prompt caching save?

4. **Multilingual:** Encode the same sentence in 5 languages using the LLaMA tokenizer. Plot tokens-per-character for each language. What does this tell you about multilingual model performance?

5. **Vocabulary overlap:** Find 10 tokens that exist in GPT-2's vocabulary but not in LLaMA's (or vice versa). What patterns do you notice?

---

## 9. Key Takeaways

| Concept | Summary |
|---|---|
| **Tokenization** | Converts raw text → integer IDs before any model computation |
| **BPE** | Learns frequent subword merges; guarantees zero OOV |
| **Vocab size** | GPT-2: 50K · LLaMA/Mistral: 32K (but byte-fallback) |
| **Context = tokens** | 4096 tokens ≠ 4096 characters |
| **Cost = tokens** | Every API call is billed by token count |
| **Failure modes** | Arithmetic, spelling, multilingual gaps trace back to tokenization |

